# CenpAI plate detector v0 (phase 1)

Trains a single-class licence-plate detector on **License Plate Detection PH BB v5** (Roboflow Universe, CC BY 4.0, attribution: Alyssa Marie), exports it to ONNX at 640 and 416, and picks the confidence threshold on the **validation** split.

**Before running** (Colab, GPU runtime = T4): add two secrets with the key icon on the left and switch on *Notebook access* for each:
- `ROBOFLOW_API_KEY` (required)
- `HF_TOKEN` (optional; if present the weights are also uploaded to a model repo on your Hugging Face account)

Nothing in this notebook contains a key. Every number it prints is measured in the run; the last cell collects them into one table.

In [ ]:
# 1. Setup
!pip -q install ultralytics roboflow onnx onnxslim onnxruntime
import ultralytics, roboflow, torch, onnxruntime
print("ultralytics", ultralytics.__version__)
print("roboflow", getattr(roboflow, "__version__", "unknown"))
print("torch", torch.__version__)
print("onnxruntime", onnxruntime.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch runtime to T4 GPU)")

from google.colab import drive, userdata
drive.mount("/content/drive")

from pathlib import Path
ROOT = Path("/content/drive/MyDrive/CenpAI")
DATA = ROOT / "data"
RUNS = ROOT / "runs"
RELEASE = ROOT / "release"
for p in (DATA, RUNS, RELEASE):
    p.mkdir(parents=True, exist_ok=True)

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

print("ROBOFLOW_API_KEY set:", bool(secret("ROBOFLOW_API_KEY")))
print("HF_TOKEN set:", bool(secret("HF_TOKEN")))

In [ ]:
# 2. Download (YOLOv8 format). Drive keeps ONE zip of the dataset; the files themselves live on local disk,
# because reading thousands of small files through the Drive mount takes many minutes.
import shutil, zipfile
from roboflow import Roboflow

RAW = Path("/content/work/raw")
RAW_ZIP = DATA / "license-plate-detection-ph-bb-v5-yolov8.zip"
if (RAW / "data.yaml").is_file():
    print("Dataset already on local disk:", RAW)
elif RAW_ZIP.is_file():
    shutil.copy(RAW_ZIP, "/content/raw.zip")
    with zipfile.ZipFile("/content/raw.zip") as z:
        z.extractall(RAW)
    print("Restored from the zip on Drive:", RAW_ZIP)
else:
    rf_key = secret("ROBOFLOW_API_KEY")
    assert rf_key, "Add ROBOFLOW_API_KEY to Colab secrets and enable Notebook access."
    rf = Roboflow(api_key=rf_key)
    project = rf.workspace("alyssa-marie-jcsyy").project("license-plate-detection-ph-bb")
    project.version(5).download("yolov8", location=str(RAW))
    shutil.make_archive("/content/raw", "zip", RAW)
    shutil.copy("/content/raw.zip", RAW_ZIP)
    print("Downloaded and cached on Drive as", RAW_ZIP)
print(sorted(p.name for p in RAW.iterdir()))

In [ ]:
# 3. Label check: keep an image only if it decodes and every label row is valid
import math, shutil, cv2
from collections import Counter

LOCAL = Path("/content/work/filtered")   # local disk: much faster to train from than Drive
SPLITS = {"train": "train", "val": "valid", "test": "test"}   # our name -> folder name in the download
PAPER = {"train": (4335, 4599), "val": (418, 445), "test": (203, 226)}
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def check_label(path):
    # returns (ok, n_boxes, reason)
    if not path.is_file():
        return False, 0, "missing label file"
    n = 0
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            return False, 0, "row without 5 fields"
        try:
            cls, cx, cy, w, h = (float(x) for x in parts)
        except ValueError:
            return False, 0, "non-numeric field"
        if not all(math.isfinite(v) for v in (cls, cx, cy, w, h)):
            return False, 0, "non-finite value"
        if cls != 0:
            return False, 0, "class not 0"
        if not (0 <= cx <= 1 and 0 <= cy <= 1):
            return False, 0, "centre outside 0..1"
        if not (w > 0 and h > 0):
            return False, 0, "non-positive size"
        n += 1
    return True, n, ""

if LOCAL.exists():
    shutil.rmtree(LOCAL)

COUNTS = {}
for split, folder in SPLITS.items():
    imgs = sorted(p for p in (RAW / folder / "images").iterdir() if p.suffix.lower() in IMG_EXT)
    (LOCAL / folder / "images").mkdir(parents=True)
    (LOCAL / folder / "labels").mkdir(parents=True)
    kept = boxes = negatives = 0
    dropped = Counter()
    for img in imgs:
        if cv2.imread(str(img)) is None:
            dropped["image does not decode"] += 1
            continue
        lab = RAW / folder / "labels" / (img.stem + ".txt")
        ok, n, why = check_label(lab)
        if not ok:
            dropped[why] += 1
            continue
        shutil.copy2(img, LOCAL / folder / "images" / img.name)
        shutil.copy2(lab, LOCAL / folder / "labels" / lab.name)
        kept += 1
        boxes += n
        negatives += (n == 0)
    COUNTS[split] = dict(raw_images=len(imgs), images=kept, boxes=boxes, negatives=negatives, dropped=dict(dropped))

DATA_YAML = LOCAL / "data.yaml"
DATA_YAML.write_text(
    f"path: {LOCAL}\ntrain: train/images\nval: valid/images\ntest: test/images\nnc: 1\nnames:\n  0: license_plate\n"
)

print(f"{'split':6} {'raw imgs':>9} {'images':>8} {'boxes':>7} {'empty':>6} | {'paper imgs':>10} {'paper boxes':>11}")
for s, c in COUNTS.items():
    print(f"{s:6} {c['raw_images']:>9} {c['images']:>8} {c['boxes']:>7} {c['negatives']:>6} | {PAPER[s][0]:>10} {PAPER[s][1]:>11}")
tot_raw = sum(c["raw_images"] for c in COUNTS.values())
tot_i = sum(c["images"] for c in COUNTS.values()); tot_b = sum(c["boxes"] for c in COUNTS.values())
print(f"{'total':6} {tot_raw:>9} {tot_i:>8} {tot_b:>7} {'':>6} | {4956:>10} {5270:>11}")
for s, c in COUNTS.items():
    if c["dropped"]:
        print("excluded in", s, c["dropped"])
print("Differences from the paper's counts are reported as they are: this is the v5 export as it exists today.")

In [ ]:
# Helpers: threshold sweep, evaluation, ONNX export (used by every later cell)
import hashlib, json, time
import numpy as np
from ultralytics import YOLO

RESULTS = []   # one dict per evaluated model; the last cell prints them all

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def load_gt(label_path):
    lines = [l for l in label_path.read_text().splitlines() if l.strip()]
    if not lines:
        return np.zeros((0, 4))
    a = np.array([[float(v) for v in l.split()] for l in lines])
    cx, cy, w, h = a[:, 1], a[:, 2], a[:, 3], a[:, 4]
    return np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=1)

def iou_many(box, gts):
    xx1 = np.maximum(box[0], gts[:, 0]); yy1 = np.maximum(box[1], gts[:, 1])
    xx2 = np.minimum(box[2], gts[:, 2]); yy2 = np.minimum(box[3], gts[:, 3])
    inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)
    a = (box[2] - box[0]) * (box[3] - box[1])
    b = (gts[:, 2] - gts[:, 0]) * (gts[:, 3] - gts[:, 1])
    return inter / np.maximum(a + b - inter, 1e-12)

def collect_predictions(weights, split, imgsz, is_pt):
    # Predict once at a very low confidence. Each prediction (highest score first) is matched to at most one
    # unused ground-truth box at IoU >= 0.50. Matching a prefix of the score-sorted list gives the same result
    # as the first part of matching the whole list, so one pass serves every threshold.
    folder = SPLITS[split]
    imgs = sorted(p for p in (LOCAL / folder / "images").iterdir() if p.suffix.lower() in IMG_EXT)
    model = YOLO(str(weights), task="detect")
    kw = dict(imgsz=imgsz, conf=0.01, iou=0.5, verbose=False, stream=True, device=0 if is_pt else "cpu")
    if is_pt:
        kw["batch"] = 16
    scores, matched, n_gt = [], [], 0
    for path, r in zip(imgs, model.predict([str(p) for p in imgs], **kw)):
        gts = load_gt(LOCAL / folder / "labels" / (path.stem + ".txt"))
        n_gt += len(gts)
        if r.boxes is None or len(r.boxes) == 0:
            continue
        sc = r.boxes.conf.cpu().numpy()
        order = np.argsort(-sc)
        sc = sc[order]; xy = r.boxes.xyxyn.cpu().numpy()[order]   # IoU is unchanged by per-axis scaling
        used = np.zeros(len(gts), dtype=bool)
        for box, s in zip(xy, sc):
            ok = False
            if len(gts):
                ious = iou_many(box, gts)
                ious[used] = -1
                j = int(ious.argmax())
                if ious[j] >= 0.5:
                    used[j] = True
                    ok = True
            scores.append(float(s)); matched.append(ok)
    return np.array(scores), np.array(matched, dtype=bool), n_gt

def counts_at(P, t):
    scores, matched, n_gt = P
    sel = scores >= t
    tp = int((sel & matched).sum()); fp = int((sel & ~matched).sum()); fn = n_gt - tp
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / n_gt if n_gt else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return dict(conf=round(float(t), 2), tp=tp, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1)

def choose_threshold(P):
    # highest confidence that keeps recall >= 0.95; if none does, the best F1 (ties go to the higher confidence)
    rows = [counts_at(P, t) for t in np.arange(0.01, 0.99, 0.01)]
    ok = [r for r in rows if r["recall"] >= 0.95]
    if ok:
        return max(ok, key=lambda r: r["conf"]), "highest conf with recall >= 0.95"
    best = max(rows, key=lambda r: (round(r["f1"], 6), r["conf"]))
    return best, "no threshold reached recall 0.95; best F1"

def evaluate(weights, imgsz, label, is_pt):
    # threshold from validation only, then test at that threshold, plus library mAP on test
    v, rule = choose_threshold(collect_predictions(weights, "val", imgsz, is_pt))
    test = counts_at(collect_predictions(weights, "test", imgsz, is_pt), v["conf"])
    m = YOLO(str(weights), task="detect").val(
        data=str(DATA_YAML), split="test", imgsz=imgsz, batch=16 if is_pt else 1,
        device=0 if is_pt else "cpu", plots=False, verbose=False)
    res = dict(model=label, imgsz=imgsz, conf=v["conf"], rule=rule,
               val_precision=v["precision"], val_recall=v["recall"], val_f1=v["f1"],
               test_precision=test["precision"], test_recall=test["recall"], test_f1=test["f1"],
               tp=test["tp"], fp=test["fp"], fn=test["fn"],
               map50=float(m.box.map50), map50_95=float(m.box.map))
    RESULTS.append(res)
    print(f"[{label} @ {imgsz}] conf={res['conf']} ({rule})")
    print(f"  validation: P={v['precision']:.4f} R={v['recall']:.4f} F1={v['f1']:.4f}")
    print(f"  test: P={res['test_precision']:.4f} R={res['test_recall']:.4f} F1={res['test_f1']:.4f} "
          f"TP={res['tp']} FP={res['fp']} FN={res['fn']}  mAP50={res['map50']:.4f} mAP50-95={res['map50_95']:.4f}")
    return res

def export_release(best_pt, out_dir, tag):
    # ONNX at 640 and 416, threshold re-picked on validation for each file, MODEL.json beside each
    import onnxruntime as ort
    out = {}
    for s in (640, 416):
        d = Path(out_dir) / f"plate-{s}"
        d.mkdir(parents=True, exist_ok=True)
        exported = YOLO(str(best_pt)).export(
            format="onnx", imgsz=s, opset=12, simplify=True, dynamic=False, half=False, nms=False)
        dst = d / f"plate_{s}.onnx"
        shutil.copy(exported, dst)
        sess = ort.InferenceSession(str(dst), providers=["CPUExecutionProvider"])
        in_shape, out_shape = sess.get_inputs()[0].shape, sess.get_outputs()[0].shape
        print(f"{dst.name}: input {in_shape}, output {out_shape}")
        assert in_shape[2] == s and in_shape[3] == s and len(out_shape) == 3 and out_shape[1] == 5, "unexpected ONNX shape"
        res = evaluate(dst, s, f"{tag} onnx", is_pt=False)
        meta = dict(file=dst.name, sha256=sha256_file(dst), imgsz=s, conf=res["conf"], iou=0.5,
                    names=["license_plate"], version=f"{tag}-{s}")
        (d / "MODEL.json").write_text(json.dumps(meta, indent=2) + "\n")
        print("  wrote", d / "MODEL.json")
        out[s] = res
    return out

## 4. Smoke run

3 epochs of `yolo11n.pt`, then exported and saved to `release/smoke/` on Drive. This unblocks the web app; download that folder first.

In [ ]:
RUN_SMOKE = True
smoke_dir = RELEASE / "smoke"
best = RUNS / "yolo11n-smoke" / "smoke" / "weights" / "best.pt"
if RUN_SMOKE and not (smoke_dir / "plate-416" / "MODEL.json").is_file():
    if best.is_file():
        print("Smoke weights already trained, reusing:", best)
    else:
        t0 = time.time()
        YOLO("yolo11n.pt").train(data=str(DATA_YAML), epochs=3, imgsz=640, batch=16, seed=0, device=0,
                                 project=str(RUNS / "yolo11n-smoke"), name="smoke", exist_ok=True)
        print(f"smoke training wall time: {(time.time() - t0) / 60:.1f} min")
    export_release(best, smoke_dir, "v0-smoke-yolo11n")
    shutil.copy(best, smoke_dir / "best.pt")
else:
    print("Smoke run skipped or already on Drive:", smoke_dir)

## 5. Full run

Set `MODEL` to `yolov8n.pt` in one Colab session and `yolo11n.pt` in the other. Up to 50 epochs, patience 10. If the session disconnects, run all cells again: it resumes from `last.pt` on Drive.

In [ ]:
MODEL = "yolov8n.pt"  #@param ["yolov8n.pt", "yolo11n.pt"]
STEM = Path(MODEL).stem
FULL_DIR = RUNS / STEM / "full"
LAST = FULL_DIR / "weights" / "last.pt"
BEST = FULL_DIR / "weights" / "best.pt"

if LAST.is_file():
    print("Found last.pt, resuming:", LAST)
    try:
        YOLO(str(LAST)).train(resume=True)
    except Exception as e:
        if "finished" in str(e).lower() or "nothing to resume" in str(e).lower():
            print("Training had already finished:", e)
        else:
            raise
else:
    YOLO(MODEL).train(data=str(DATA_YAML), epochs=50, patience=10, imgsz=640, batch=16, seed=0, device=0,
                      project=str(RUNS / STEM), name="full", exist_ok=True)
assert BEST.is_file(), "best.pt missing"
print("best weights:", BEST)

In [ ]:
# 6 and 7. Threshold on validation, then test at that threshold (best.pt at 640)
RESULTS.clear()
res_pt = evaluate(BEST, 640, f"{STEM} best.pt", is_pt=True)

In [ ]:
# 8. Export ONNX (640 and 416), threshold repeated per file, MODEL.json beside each
TAG = f"v0-{STEM}"
REL = RELEASE / STEM
onnx_res = export_release(BEST, REL, TAG)

In [ ]:
# 9. Save to Drive; optional Hugging Face upload
import pandas as pd
shutil.copy(BEST, REL / "best.pt")
shutil.copy(FULL_DIR / "results.csv", REL / "results.csv")

saved_nb = False
for cand in Path("/content/drive/MyDrive").glob("Colab Notebooks/train_detector_v0*.ipynb"):
    shutil.copy(cand, REL / "train_detector_v0.ipynb")
    saved_nb = True
    break
if not saved_nb:
    print("Notebook file not found under Colab Notebooks/. Use File > Save a copy in Drive to keep it with the run.")

hub_token = secret("HF_TOKEN")
if hub_token:
    from huggingface_hub import HfApi
    api = HfApi(token=hub_token)
    owner = api.whoami()["name"]
    repo_id = f"{owner}/cenpai-plate-detector-v0-{STEM}"
    api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(REL), repo_id=repo_id, repo_type="model")
    print("uploaded to", repo_id, "(private; change visibility on the Hub if you want it public)")
else:
    print("HF_TOKEN not set; skipped the Hugging Face upload.")

df = pd.read_csv(FULL_DIR / "results.csv")
df.columns = [c.strip() for c in df.columns]
fit = 0.1 * df["metrics/mAP50(B)"] + 0.9 * df["metrics/mAP50-95(B)"]
TRAIN_INFO = dict(epochs_run=len(df), best_epoch=int(df["epoch"].iloc[int(fit.values.argmax())]))
print(TRAIN_INFO)

In [ ]:
# 10. Summary: one Markdown table of everything measured in this session
def f4(x):
    return f"{x:.4f}"

print(f"Run: {MODEL}, epochs run {TRAIN_INFO['epochs_run']}, best epoch {TRAIN_INFO['best_epoch']} (as numbered in results.csv)")
print()
print("Dataset (this download vs paper)")
print("| split | images | boxes | empty images | paper images | paper boxes |")
print("|---|---|---|---|---|---|")
for s, c in COUNTS.items():
    print(f"| {s} | {c['images']} | {c['boxes']} | {c['negatives']} | {PAPER[s][0]} | {PAPER[s][1]} |")
print()
print("Models")
print("| model | imgsz | conf | val P | val R | test P | test R | test F1 | TP | FP | FN | mAP@0.50 | mAP@0.50:0.95 |")
print("|---|---|---|---|---|---|---|---|---|---|---|---|---|")
for r in RESULTS:
    print(f"| {r['model']} | {r['imgsz']} | {r['conf']} | {f4(r['val_precision'])} | {f4(r['val_recall'])} | "
          f"{f4(r['test_precision'])} | {f4(r['test_recall'])} | {f4(r['test_f1'])} | {r['tp']} | {r['fp']} | {r['fn']} | "
          f"{f4(r['map50'])} | {f4(r['map50_95'])} |")
print()
print("Files (sha256 from MODEL.json)")
for s in (640, 416):
    m = json.loads((REL / f"plate-{s}" / "MODEL.json").read_text())
    print(f"- plate-{s}: {m['file']} {m['sha256']} conf={m['conf']} version={m['version']}")
print()
print("Library mAP is computed with the Ultralytics validator on the test split; precision, recall and F1 use the matching rule in this notebook.")